# Módulo 06: MLflow Tracking

## Contenido del Módulo

1. **Introducción a MLflow**
   - ¿Qué es MLflow?
   - Componentes de MLflow
   - MLflow en Databricks

2. **MLflow Tracking**
   - Experimentos y Runs
   - Logging: Parámetros, Métricas, Artifacts
   - Modelos y Tags

3. **MLflow UI**
   - Navegación y búsqueda
   - Comparación de runs
   - Visualizaciones

4. **Integración con Databricks**
   - AutoML + MLflow
   - Feature Store + MLflow
   - Notebooks + MLflow

5. **Mejores Prácticas**
   - Organización de experimentos
   - Logging efectivo
   - Reproducibilidad

---

**Objetivo**: Dominar el tracking de experimentos ML para garantizar reproducibilidad, comparabilidad y trazabilidad en el ciclo de vida de modelos.

# 1. Introducción a MLflow

## ¿Qué es MLflow?

**MLflow** es una plataforma open-source para gestionar el ciclo de vida completo de Machine Learning:
- Experimentación y desarrollo
- Reproducción de resultados
- Deployment y productización
- Monitoreo y gobernanza

## Componentes de MLflow

MLflow consta de **4 componentes principales**:

| Componente | Descripción | Uso |
|------------|-------------|-----|
| **MLflow Tracking** | Registro de experimentos, parámetros, métricas, artifacts | 🔵 **Foco de este módulo** |
| **MLflow Projects** | Formato para empaquetar código ML reproducible | Proyectos reutilizables |
| **MLflow Models** | Formato estándar para empaquetar modelos | Deployment multi-framework |
| **MLflow Registry** | Repositorio centralizado de modelos | Versionado y lifecycle |

## MLflow Tracking: El Corazón de MLflow

**MLflow Tracking** es el componente más usado. Permite:

```
┌─────────────────────────────────────────┐
│          MLFLOW TRACKING                 │
├─────────────────────────────────────────┤
│  📋 Parámetros (learning_rate, epochs)  │
│  📊 Métricas (accuracy, loss, f1)      │
│  💾 Artifacts (modelos, plots, data)    │
│  🏷️ Tags (metadata, descripción)          │
│  📄 Código fuente (automático)           │
│  🕰️ Timestamps (inicio, fin, duración)   │
└─────────────────────────────────────────┘
```

## MLflow en Databricks

En Databricks, MLflow está:
- **✅ Preinstalado** y preconfigurado
- **✅ Integrado** con notebooks, AutoML, Feature Store
- **✅ Gestionado** (no requiere setup de infraestructura)
- **✅ Conectado** con Unity Catalog para gobernanza
- **✅ Accesible** vía UI web interactiva

```python
# MLflow en Databricks está listo para usar
import mlflow

# Automáticamente conectado al workspace
print(f"MLflow version: {mlflow.__version__}")
print(f"Tracking URI: {mlflow.get_tracking_uri()}")
```

## ¿Por Qué Necesitamos MLflow Tracking?

### Sin MLflow

```python
# Experimento 1
model = train_model(lr=0.01, epochs=10)
accuracy = 0.85  # ¿Cómo obtuvo este resultado?

# Experimento 2 (al día siguiente)
model = train_model(lr=0.001, epochs=20)  
accuracy = 0.87  # ¿Qué parámetros usé?

# ❌ Problemas:
# - No puedo reproducir resultados
# - No sé qué parámetros dieron mejor accuracy
# - No tengo el modelo guardado
# - No puedo comparar experimentos
```

### Con MLflow

```python
import mlflow

with mlflow.start_run():
    mlflow.log_param("learning_rate", 0.01)
    mlflow.log_param("epochs", 10)
    
    model = train_model(lr=0.01, epochs=10)
    accuracy = evaluate(model)
    
    mlflow.log_metric("accuracy", accuracy)
    mlflow.sklearn.log_model(model, "model")

# ✅ Ventajas:
# - Todos los parámetros registrados
# - Métricas historizadas
# - Modelo versionado
# - Reproducible
# - Comparable en UI
```

# 2. Conceptos Clave de MLflow Tracking

## Jerarquía de MLflow

```
Workspace
  │
  ├── Experiment 1: "Customer Churn"
  │     ├── Run 1 (Random Forest, lr=0.01)
  │     ├── Run 2 (XGBoost, lr=0.001)
  │     └── Run 3 (Logistic Regression)
  │
  ├── Experiment 2: "Sales Forecasting"
  │     ├── Run 1 (ARIMA)
  │     └── Run 2 (Prophet)
  │
  └── Experiment 3: "Image Classification"
        ├── Run 1 (CNN)
        └── Run 2 (Transfer Learning)
```

## 1. Experiment (Experimento)

Un **Experiment** agrupa múltiples runs relacionados:
- Un proyecto o problema de negocio
- Diferentes enfoques para resolver el mismo problema
- Iteraciones sobre el mismo modelo

**Ejemplo**: "Predicción de Churn Bancario"

## 2. Run (Ejecución)

Un **Run** es una ejecución individual de código ML:
- Un entrenamiento de modelo
- Una configuración de hiperparámetros
- Un experimento específico

**Cada run registra**:

| Elemento | Descripción | Ejemplo |
|----------|-------------|--------|
| **Parámetros** | Inputs configurables (inmutables) | `learning_rate=0.01` |
| **Métricas** | Outputs medibles (mutables, históricos) | `accuracy=0.85` |
| **Artifacts** | Archivos de salida | Modelo, plots, CSV |
| **Tags** | Metadata clave-valor | `environment=production` |
| **Código** | Versión del código (Git commit) | `commit_hash=abc123` |
| **Timestamps** | Inicio, fin, duración | `start_time`, `end_time` |

## 3. Parámetros vs. Métricas

### Parámetros (Immutable Inputs)

- **Qué son**: Configuraciones del modelo/entrenamiento
- **Características**:
  - Definidos **antes** del entrenamiento
  - **Inmutables** durante el run
  - Solo **valores escalares** (int, float, string)

```python
mlflow.log_param("learning_rate", 0.01)
mlflow.log_param("max_depth", 10)
mlflow.log_param("algorithm", "random_forest")
```

### Métricas (Mutable Outputs)

- **Qué son**: Mediciones del rendimiento
- **Características**:
  - Calculadas **durante/después** del entrenamiento
  - **Mutables** (pueden actualizarse en cada epoch)
  - Solo **valores numéricos** (int, float)
  - Soportan **historial** (para gráficas de entrenamiento)

```python
mlflow.log_metric("accuracy", 0.85)
mlflow.log_metric("f1_score", 0.82)

# Logging iterativo (por epoch)
for epoch in range(10):
    loss = train_epoch()
    mlflow.log_metric("loss", loss, step=epoch)
```

## 4. Artifacts

**Artifacts** son archivos generados durante el run:

| Tipo | Ejemplos |
|------|----------|
| **Modelos** | `model.pkl`, `model.h5` |
| **Plots** | `confusion_matrix.png`, `roc_curve.png` |
| **Data** | `predictions.csv`, `feature_importance.json` |
| **Configs** | `config.yaml`, `requirements.txt` |
| **Notebooks** | `training_notebook.html` |

```python
# Log modelo
mlflow.sklearn.log_model(model, "model")

# Log archivo
mlflow.log_artifact("confusion_matrix.png")

# Log directorio completo
mlflow.log_artifacts("output_folder/")
```

## 5. Tags

**Tags** son metadata adicional en formato clave-valor:

```python
mlflow.set_tag("environment", "production")
mlflow.set_tag("team", "data-science")
mlflow.set_tag("model_type", "ensemble")
mlflow.set_tag("notes", "Mejor modelo hasta ahora")
```

**Uso común**:
- Filtrado y búsqueda en UI
- Organización de experimentos
- Anotaciones y documentación

In [0]:
# 3. Ejemplo Completo de Run

import mlflow
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

# Crear datos de ejemplo
X, y = make_classification(n_samples=1000, n_features=20, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Iniciar un run de MLflow
with mlflow.start_run(run_name="ejemplo_completo_rf") as run:
    
    # 1. Log parámetros
    params = {
        "n_estimators": 100,
        "max_depth": 10,
        "min_samples_split": 5,
        "random_state": 42
    }
    mlflow.log_params(params)
    
    # 2. Entrenar modelo
    model = RandomForestClassifier(**params)
    model.fit(X_train, y_train)
    
    # 3. Evaluar y log métricas
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    
    metrics = {
        "accuracy": accuracy_score(y_test, y_pred),
        "f1_score": f1_score(y_test, y_pred),
        "roc_auc": roc_auc_score(y_test, y_proba)
    }
    mlflow.log_metrics(metrics)
    
    # 4. Log modelo
    mlflow.sklearn.log_model(model, "model")
    
    # 5. Log tags
    mlflow.set_tag("model_type", "random_forest")
    mlflow.set_tag("dataset", "synthetic_classification")
    mlflow.set_tag("framework", "sklearn")
    
    # Imprimir información del run
    print(f"✅ Run completado")
    print(f"\nRun ID: {run.info.run_id}")
    print(f"Experiment ID: {run.info.experiment_id}")
    print(f"\nMétricas:")
    for metric, value in metrics.items():
        print(f"  {metric}: {value:.4f}")

# 4. MLflow UI - Navegación y Visualización

## Acceder a la UI de MLflow

En Databricks, hay **3 formas** de acceder a MLflow UI:

### 1. Desde el Notebook

Después de ejecutar un run:
```python
with mlflow.start_run() as run:
    # ... tu código ...
    print(f"Ver run: {run.info.artifact_uri}")
```

Haz clic en el icono de MLflow 📋 en la barra lateral del notebook.

### 2. Desde el Menú Principal

* **Machine Learning** → **Experiments**
* Ver todos los experimentos del workspace

### 3. URL Directa

```
https://<workspace-url>/#mlflow/experiments/<experiment_id>
```

---

## Vistas Principales de la UI

### Vista de Experimentos

Muestra todos los runs de un experimento:

| Columna | Descripción |
|---------|-------------|
| **Run Name** | Nombre del run |
| **Created** | Timestamp de creación |
| **Duration** | Tiempo de ejecución |
| **Source** | Notebook/script origen |
| **User** | Usuario que ejecutó el run |
| **Metrics** | Métricas clave (configurables) |
| **Params** | Parámetros (configurables) |

### Vista de Run Individual

Detalles de un run específico:

```
┌──────────────────────────────────┐
│        RUN DETAILS              │
├──────────────────────────────────┤
│ 📋 Parameters              │
│   - learning_rate: 0.01      │
│   - max_depth: 10            │
├──────────────────────────────────┤
│ 📊 Metrics                  │
│   - accuracy: 0.85           │
│   - f1_score: 0.82           │
├──────────────────────────────────┤
│ 💾 Artifacts                │
│   - model/                   │
│   - confusion_matrix.png     │
├──────────────────────────────────┤
│ 🏷️ Tags                      │
│   - model_type: rf           │
│   - environment: dev         │
└──────────────────────────────────┘
```

---

## Funcionalidades Clave

### 1. Comparación de Runs

* Selecciona múltiples runs (checkbox)
* Click en **"Compare"**
* Ve diferencias en:
  - Parámetros
  - Métricas
  - Visualizaciones paralelas

### 2. Búsqueda y Filtrado

```sql
-- Buscar por métrica
metrics.accuracy > 0.8

-- Buscar por parámetro
params.learning_rate = "0.01"

-- Buscar por tag
tags.model_type = "random_forest"

-- Combinaciones
metrics.accuracy > 0.8 AND params.max_depth < 15
```

### 3. Gráficas de Métricas

Para métricas con historial (`step`):
* Gráfica de loss vs. epoch
* Comparación de curvas de aprendizaje
* Identificación de overfitting

### 4. Descarga de Artifacts

* Click en cualquier artifact
* Visualización inline (imágenes, JSON, CSV)
* Descarga local

### 5. Registro de Modelos

Desde un run:
* **"Register Model"**
* Crear modelo nuevo o agregar versión
* Integración con Model Registry

In [0]:
# 5. Comparación de Múltiples Runs

import mlflow
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Datos
X, y = make_classification(n_samples=1000, n_features=20, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

# Experimento para comparar diferentes configuraciones
mlflow.set_experiment("/Users/cortega@uda.edu.ar/comparacion_hiperparametros")

print("Ejecutando múltiples runs con diferentes hiperparámetros...\n")

# Configuraciones a probar
configs = [
    {"n_estimators": 50, "max_depth": 5},
    {"n_estimators": 100, "max_depth": 10},
    {"n_estimators": 200, "max_depth": 15},
    {"n_estimators": 100, "max_depth": 20},
]

for i, config in enumerate(configs, 1):
    with mlflow.start_run(run_name=f"config_{i}"):
        # Log parámetros
        mlflow.log_params(config)
        
        # Entrenar
        model = RandomForestClassifier(**config, random_state=42)
        model.fit(X_train, y_train)
        
        # Evaluar
        accuracy = accuracy_score(y_test, model.predict(X_test))
        mlflow.log_metric("accuracy", accuracy)
        
        # Tag
        mlflow.set_tag("config_id", f"config_{i}")
        
        print(f"Config {i}: n_estimators={config['n_estimators']}, "
              f"max_depth={config['max_depth']} → Accuracy={accuracy:.4f}")

print("\n✅ Runs completados")
print("\n👉 Ve a MLflow UI para comparar:")
print("   1. Selecciona los 4 runs")
print("   2. Click en 'Compare'")
print("   3. Analiza Parallel Coordinates Plot")

# 6. Mejores Prácticas de MLflow Tracking

## Organización de Experimentos

### ✅ Buenas Prácticas

| Práctica | Descripción |
|----------|-------------|
| **Nombres descriptivos** | `customer_churn_v2` en vez de `experiment_1` |
| **Jerarquía por proyecto** | `/Shared/banking/credit_risk/model_v2` |
| **Un experimento por modelo** | No mezclar clasificación con regresión |
| **Documentar con tags** | Añadir contexto en cada run |
| **Versionado semántico** | `v1.0`, `v1.1`, `v2.0` en tags |

### ❌ Anti-patrones

* Experimentos con nombres genéricos (`test`, `experiment1`)
* Mixing unrelated runs en un solo experimento
* No documentar parámetros críticos
* No usar tags para filtrado

---

## Logging Efectivo

### Qué Loggear

| Tipo | ¿Cuándo loggear? | Ejemplos |
|------|----------------|----------|
| **Parámetros** | Siempre | Hiperparámetros, configuraciones |
| **Métricas** | Siempre | Accuracy, loss, F1, AUC |
| **Modelo** | Modelos finales | Mejores modelos, checkpoints |
| **Artifacts** | Visualizaciones importantes | Confusion matrix, ROC curves |
| **Tags** | Para organización | Environment, version, status |

### Parámetros Esenciales a Loggear

```python
# Hiperparámetros del modelo
mlflow.log_param("learning_rate", lr)
mlflow.log_param("batch_size", batch_size)
mlflow.log_param("epochs", epochs)

# Parámetros de datos
mlflow.log_param("train_size", len(X_train))
mlflow.log_param("test_size", len(X_test))
mlflow.log_param("n_features", X_train.shape[1])

# Parámetros de preprocessing
mlflow.log_param("scaler", "StandardScaler")
mlflow.log_param("handle_missing", "mean_imputation")

# Configuración del entorno
mlflow.log_param("framework", "sklearn")
mlflow.log_param("python_version", sys.version)
```

### Métricas Esenciales

```python
# Métricas de rendimiento
mlflow.log_metric("train_accuracy", train_acc)
mlflow.log_metric("val_accuracy", val_acc)
mlflow.log_metric("test_accuracy", test_acc)

# Métricas de negocio (si aplica)
mlflow.log_metric("expected_revenue_increase", revenue_gain)
mlflow.log_metric("false_positive_cost", fp_cost)

# Metadata de entrenamiento
mlflow.log_metric("training_time_seconds", duration)
mlflow.log_metric("model_size_mb", model_size)
```

---

## Reproducibilidad

### Elementos Clave

1. **Parámetros completos**: Todos los hiperparámetros relevantes
2. **Seeds**: Random seeds para reproducibilidad
3. **Environment**: Versiones de librerías
4. **Data**: Versión/snapshot de datos usados
5. **Código**: Git commit hash

### Ejemplo Completo Reproducible

```python
import mlflow
import sys
import sklearn
import numpy as np

with mlflow.start_run():
    # 1. Log seeds
    seed = 42
    mlflow.log_param("random_seed", seed)
    np.random.seed(seed)
    
    # 2. Log versiones
    mlflow.log_param("sklearn_version", sklearn.__version__)
    mlflow.log_param("python_version", sys.version)
    
    # 3. Log data version
    mlflow.log_param("data_version", "2024-01-15")
    mlflow.log_param("data_source", "main.features.customer_features")
    
    # 4. Log git commit (si está disponible)
    try:
        import git
        repo = git.Repo(search_parent_directories=True)
        mlflow.set_tag("git_commit", repo.head.object.hexsha)
    except:
        pass
    
    # 5. Entrenar modelo
    model = train_model(seed=seed)
    
    # 6. Log modelo con signature
    mlflow.sklearn.log_model(
        model, 
        "model",
        signature=mlflow.models.infer_signature(X_train, y_train)
    )
```

---

## Integración con Otros Componentes

### AutoML + MLflow

Databricks AutoML automáticamente:
* Crea experimento MLflow
* Registra todos los runs
* Loggea parámetros, métricas, modelos
* Genera notebooks con código MLflow

### Feature Store + MLflow

Modelos entrenados con Feature Store:
* Metadata de features en MLflow
* Linaje de features
* Serving automático con feature lookup

### Notebooks + MLflow

En Databricks notebooks:
* MLflow tracking automático
* Experimento por notebook (por defecto)
* UI integrada en sidebar
* Link directo a runs en output cells

In [0]:
# 7. Búsqueda y Consulta de Runs Programaticamente

import mlflow
from mlflow.tracking import MlflowClient

client = MlflowClient()

# 1. Buscar runs por métrica
print("±±=" * 30)
print("BUSCAR RUNS CON ACCURACY > 0.80")
print("=" * 60)

runs = mlflow.search_runs(
    filter_string="metrics.accuracy > 0.80",
    order_by=["metrics.accuracy DESC"],
    max_results=5
)

if not runs.empty:
    print(runs[['run_id', 'params.n_estimators', 'metrics.accuracy']].head())
else:
    print("No se encontraron runs con accuracy > 0.80")

# 2. Buscar por parámetro
print("\n" + "=" * 60)
print("BUSCAR RUNS CON max_depth = 10")
print("=" * 60)

runs_by_param = mlflow.search_runs(
    filter_string="params.max_depth = '10'"
)

if not runs_by_param.empty:
    print(f"Encontrados: {len(runs_by_param)} runs")
else:
    print("No se encontraron runs con max_depth=10")

# 3. Buscar best run
print("\n" + "=" * 60)
print("MEJOR RUN POR ACCURACY")
print("=" * 60)

best_runs = mlflow.search_runs(
    order_by=["metrics.accuracy DESC"],
    max_results=1
)

if not best_runs.empty:
    best_run = best_runs.iloc[0]
    print(f"Run ID: {best_run['run_id']}")
    print(f"Accuracy: {best_run['metrics.accuracy']:.4f}")
    if 'params.n_estimators' in best_run:
        print(f"N Estimators: {best_run['params.n_estimators']}")
    if 'params.max_depth' in best_run:
        print(f"Max Depth: {best_run['params.max_depth']}")

print("\n✅ Búsquedas completadas")

# Resumen y Conclusiones

## Conceptos Clave Aprendidos

### MLflow Tracking

✅ **Experiment**: Agrupa runs relacionados  
✅ **Run**: Ejecución individual con parámetros, métricas, artifacts  
✅ **Parámetros**: Configuraciones inmutables (inputs)  
✅ **Métricas**: Mediciones mutables (outputs)  
✅ **Artifacts**: Archivos generados (modelos, plots, data)  
✅ **Tags**: Metadata para organización y filtrado  

### MLflow UI

✅ Navegación intuitiva de experimentos y runs  
✅ Comparación visual de múltiples runs  
✅ Búsqueda y filtrado avanzado  
✅ Visualización de métricas históricas  
✅ Descarga y visualización de artifacts  

### Mejores Prácticas

✅ Nombres descriptivos para experimentos  
✅ Loggear parámetros completos  
✅ Métricas de negocio además de técnicas  
✅ Tags para organización  
✅ Reproducibilidad (seeds, versions, data)  

---

## Workflow Recomendado

```
1. Crear Experimento
   mlflow.set_experiment("/path/to/experiment")

2. Iniciar Run
   with mlflow.start_run(run_name="descriptive_name"):

3. Log Parámetros
   mlflow.log_params({...})

4. Entrenar Modelo
   model = train(...)

5. Log Métricas
   mlflow.log_metrics({...})

6. Log Modelo
   mlflow.sklearn.log_model(model, "model")

7. Log Artifacts
   mlflow.log_artifact("plot.png")

8. Tags
   mlflow.set_tag("status", "production")

9. Comparar en UI
   - Seleccionar runs
   - Comparar métricas
   - Identificar mejor modelo

10. Registrar Mejor Modelo
    - Model Registry
    - Versionado
    - Lifecycle management
```

---
## Beneficios de MLflow Tracking

| Beneficio | Impacto |
|-----------|--------|
| **Reproducibilidad** | Cualquier experimento puede repetirse exactamente |
| **Comparabilidad** | Fácil comparar múltiples enfoques |
| **Trazabilidad** | Historial completo de experimentación |
| **Colaboración** | Compartir resultados entre equipos |
| **Documentación** | Registro automático de decisiones |
| **Gobernanza** | Auditoría y compliance |

---

## Próximos Pasos

En el **notebook de práctica** aplicarás:

1. Crear experimentos organizados
2. Loggear parámetros, métricas, artifacts
3. Comparar múltiples runs
4. Buscar y filtrar runs programaticamente
5. Implementar workflow completo reproducible
6. Integrar con Feature Store y AutoML

**Continúa al notebook:** `Práctica - MLflow Tracking` 🚀